# EPCOT Guest Flow & Ride Operations

**Author:** Andrew Gentilcore  
**Dataset:** Synthetic EPCOT guest movement data (~80,000 records)  
**Purpose:** Identify patterns in crowd flow, ride wait times, and guest experience across park zones and festivals

---

### What This Notebook Covers

| # | Section | Question |
|---|---|---|
| 1 | Setup | |
| 2 | Load and Validate | What does the dataset look like? |
| 3 | Feature Engineering | |
| 4 | Zone Congestion | When and where does guest density peak? |
| 5 | Guest Flow Transitions | Which zone-to-zone paths are most traveled? |
| 6 | Ride Wait Times | Which attractions create the highest time cost? |
| 7 | Posted vs. Actual Wait Accuracy | How well do posted times reflect reality? |
| 8 | Experience Metrics | How do waits and crowds relate to fun and frustration? |
| 9 | Ride Quality Confounding | Why do popular rides show higher fun AND higher waits? |
| 10 | Ticket Type Segmentation | Do annual passholders behave differently than standard guests? |
| 11 | Festival Comparison | Does festival context shift guest experience outcomes? |
| 12 | Correlation Summary | How do all numeric variables relate? |
| 13 | Conclusion | Key takeaways and next steps |

> **Data Note:** This dataset is synthetic and built for portfolio purposes. It mirrors realistic EPCOT guest circulation patterns, including ride demand hierarchy, time-of-day crowd curves, posted vs. actual wait dynamics, and experience score drivers. No proprietary Disney data was used.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
pd.set_option('display.max_columns', None)

---
## 2. Load and Validate Data

In [ ]:
df = pd.read_csv('epcot_guest_flow_large_stable.csv', parse_dates=['event_timestamp'])

print(f'Rows:           {len(df):,}')
print(f'Unique guests:  {df["guest_id"].nunique():,}')
print(f'Date range:     {df["event_timestamp"].min().date()} → {df["event_timestamp"].max().date()}')
print(f'Ride events:    {df["ride_name"].notna().sum():,} ({df["ride_name"].notna().mean():.1%} of all events)')
print(f'Festivals:      {sorted(df["festival"].unique())}')
print(f'\nMissing values (ride fields are null for movement-only events):')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.head(5)

---
## 3. Feature Engineering

The `wait_delta` field captures the gap between actual and posted wait time. Negative values mean guests got through the queue faster than the sign said, which tends to leave a good impression.

In [ ]:
df['hour']         = df['event_timestamp'].dt.hour
df['date']         = df['event_timestamp'].dt.date
df['day_of_week']  = df['event_timestamp'].dt.day_name()
df['is_weekend']   = df['event_timestamp'].dt.dayofweek >= 5
df['ride_event']   = df['ride_name'].notna()

# Wait delta: actual minus posted (negative = shorter than posted = positive surprise for guest)
df['wait_delta'] = df['actual_wait_min'] - df['posted_wait_min']

rides = df[df['ride_event']].copy()
print(f'Ride events: {len(rides):,} across {rides["ride_name"].nunique()} attractions')

---
## 4. Zone Congestion by Time of Day

**Question:** When and where does guest density peak across EPCOT's 15 zones?

I'm counting inbound movement events per zone per hour as a proxy for accumulation. EPCOT typically opens at 9am and closes around 9pm.

In [ ]:
zone_congestion = (
    df.groupby(['to_zone', 'hour'])
      .size()
      .reset_index(name='event_count')
)

pivot_zone = zone_congestion.pivot(index='to_zone', columns='hour', values='event_count').fillna(0)
pivot_zone = pivot_zone.loc[pivot_zone.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    pivot_zone,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Inbound Movement Events'},
    ax=ax
)
ax.set_title('EPCOT Zone Congestion by Hour of Day', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Hour of Day (24h)', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Crowd level arc by hour — confirms the realistic bell curve
hourly_crowd = df.groupby('hour')['crowd_level'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_crowd['hour'], hourly_crowd['crowd_level'], marker='o', linewidth=2,
        color=sns.color_palette('muted')[3])
ax.fill_between(hourly_crowd['hour'], hourly_crowd['crowd_level'], alpha=0.15,
                color=sns.color_palette('muted')[3])
ax.set_title('Average Crowd Level by Hour', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Crowd Level (1–10)')
ax.set_xticks(hourly_crowd['hour'])
plt.tight_layout()
plt.show()

**Takeaway:** Crowd density peaks between 11am and 2pm, with a secondary bump around 7pm. World Discovery and World Nature, which house the park's most in-demand rides, show the widest and most sustained activity windows. World Showcase picks up through the afternoon as guests work their way through pavilions, which lines up with how most people actually navigate EPCOT.

---
## 5. Guest Flow Transitions

**Question:** Which zone-to-zone paths are most frequently traveled?

Looking at where guests move and in what direction can reveal corridor bottlenecks and help frame where operational changes would have the most impact.

In [ ]:
flow = (
    df.groupby(['from_zone', 'to_zone'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
      .head(15)
)

# Shorten labels for readability
def shorten(z):
    return z.replace('World Showcase ', '').replace('World ', '')

flow['path'] = flow['from_zone'].map(shorten) + ' → ' + flow['to_zone'].map(shorten)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(flow['path'], flow['count'], color=sns.color_palette('muted')[0])
ax.set_title('Top 15 Guest Flow Transitions', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Movement Events')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar in bars:
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width()):,}', va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

**Takeaway:** The busiest transitions run along the World Showcase corridor, which makes sense given how the park is laid out. Guests tend to walk sequentially through pavilions rather than jumping across the loop. The Entrance to World Discovery and Entrance to World Nature paths reflect the morning pattern of heading straight to the high-demand rides before lines build.

---
## 6. Ride Wait Time Analysis

**Question:** Which attractions create the highest average time cost for guests, and how consistent are those waits?

In [ ]:
wait_stats = (
    rides.groupby('ride_name')['actual_wait_min']
         .agg(['mean', 'median', 'std', 'count'])
         .rename(columns={'mean': 'avg_wait', 'median': 'med_wait',
                          'std': 'std_wait', 'count': 'ride_count'})
         .sort_values('avg_wait', ascending=False)
         .reset_index()
)
wait_stats.round(1)

In [ ]:
top10 = wait_stats.head(10).sort_values('avg_wait')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['ride_name'], top10['avg_wait'],
               xerr=top10['std_wait'], capsize=3,
               color=sns.color_palette('muted')[1], error_kw={'elinewidth': 1.2})
ax.set_title('Top 10 Rides by Average Actual Wait Time\n(error bars = ±1 std dev)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Average Actual Wait (Minutes)')
ax.set_ylabel('')

for bar in bars:
    ax.text(bar.get_width() + 2.5, bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.0f} min', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Takeaway:** Guardians of the Galaxy, Frozen Ever After, and Remy's Ratatouille Adventure sit at the top, which tracks with their real-world demand at EPCOT. The error bars show standard deviation, and Guardians and Frozen both carry wide ranges, meaning the wait on any given day can vary a lot depending on crowd conditions. Rides like Spaceship Earth and Gran Fiesta Tour are much more predictable, which makes them useful fallback options when the park is packed.

---
## 7. Posted vs. Actual Wait Time Accuracy

**Question:** How well do the signs reflect what guests actually experience?

`wait_delta = actual_wait - posted_wait`

A negative value means the guest got through faster than expected. A positive value means they waited longer than the sign said.

In [ ]:
delta_by_ride = (
    rides.groupby('ride_name')['wait_delta']
         .mean()
         .sort_values()
         .reset_index()
)

colors = ['#4575b4' if x < 0 else '#d73027' for x in delta_by_ride['wait_delta']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(delta_by_ride['ride_name'], delta_by_ride['wait_delta'], color=colors)
ax.axvline(0, color='black', linewidth=0.9, linestyle='--')
ax.set_title('Average Wait Delta by Ride\n(Actual − Posted Wait Time)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Minutes (negative = shorter than posted)')
ax.set_ylabel('')

legend_elements = [
    mpatches.Patch(facecolor='#4575b4', label='Actual < Posted  (positive surprise)'),
    mpatches.Patch(facecolor='#d73027', label='Actual > Posted  (expectation violation)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of wait delta across all ride events
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rides['wait_delta'].dropna(), bins=50, color=sns.color_palette('muted')[0],
        edgecolor='white', linewidth=0.4)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(rides['wait_delta'].mean(), color='red', linestyle='-', linewidth=1.2,
           label=f'Mean: {rides["wait_delta"].mean():.1f} min')
ax.set_title('Distribution of Wait Delta (Actual − Posted)', fontsize=13, fontweight='bold')
ax.set_xlabel('Wait Delta (minutes)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

**Takeaway:** On average, guests wait about 2 to 3 minutes less than posted. Parks tend to overstate times slightly to set up a positive surprise when guests exit the queue faster than expected. The distribution skews left, so most guests beat the sign. That said, there are right-tail cases where actual waits run meaningfully longer than posted, and those are the situations most likely to drive frustration.

---
## 8. Experience Metrics: Fun, Frustration, and Wait Time

**Question:** How do operational factors like wait time and crowd level relate to what guests actually feel?

In [ ]:
sample = rides.dropna(subset=['fun_score', 'frustration_score', 'actual_wait_min']).sample(6000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wait vs Frustration
sns.scatterplot(data=sample, x='actual_wait_min', y='frustration_score',
                alpha=0.25, color=sns.color_palette('muted')[3], ax=axes[0])
sns.regplot(data=sample, x='actual_wait_min', y='frustration_score',
            scatter=False, color='black', line_kws={'linewidth': 1.5}, ax=axes[0])
r, p = stats.pearsonr(sample['actual_wait_min'], sample['frustration_score'])
axes[0].set_title(f'Wait Time vs. Frustration Score\n(r = {r:.3f}, p < 0.001)',
                  fontweight='bold')
axes[0].set_xlabel('Actual Wait (Minutes)')
axes[0].set_ylabel('Frustration Score')

# Crowd level vs Frustration
sns.scatterplot(data=sample, x='crowd_level', y='frustration_score',
                alpha=0.25, color=sns.color_palette('muted')[1], ax=axes[1])
sns.regplot(data=sample, x='crowd_level', y='frustration_score',
            scatter=False, color='black', line_kws={'linewidth': 1.5}, ax=axes[1])
r2, p2 = stats.pearsonr(sample['crowd_level'], sample['frustration_score'])
axes[1].set_title(f'Crowd Level vs. Frustration Score\n(r = {r2:.3f}, p < 0.001)',
                  fontweight='bold')
axes[1].set_xlabel('Crowd Level (1–10)')
axes[1].set_ylabel('Frustration Score')

plt.suptitle('Drivers of Guest Frustration', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Crowd level vs both scores
crowd_exp = (
    df.dropna(subset=['crowd_level', 'fun_score', 'frustration_score'])
      .assign(crowd_round=lambda d: d['crowd_level'].round())
      .groupby('crowd_round')[['fun_score', 'frustration_score']]
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(crowd_exp['crowd_round'], crowd_exp['fun_score'],
        marker='o', label='Fun Score', color=sns.color_palette('muted')[2])
ax.plot(crowd_exp['crowd_round'], crowd_exp['frustration_score'],
        marker='o', label='Frustration Score', color=sns.color_palette('muted')[3])
ax.set_title('Crowd Level vs. Average Experience Scores', fontweight='bold')
ax.set_xlabel('Crowd Level (1–10)')
ax.set_ylabel('Average Score')
ax.legend()
plt.tight_layout()
plt.show()

**Takeaway:** Wait time is a strong predictor of frustration (r near 0.89), but crowd level also matters independently. Even when waits are similar, guests in denser parks report worse experiences. As crowd level increases, fun scores drop and frustration scores rise, and the gap between them shrinks. That convergence at high crowd levels suggests park density starts degrading the overall visit in ways that go beyond any single queue.

---
## 9. Ride Quality Confounding

**Question:** The aggregate data shows a positive correlation between wait time and fun score. Longer waits mean more fun? That does not make sense. What is actually going on?

This is a confounding problem. Guardians of the Galaxy has both the highest average fun score and the longest average wait. Gran Fiesta Tour has both the lowest fun score and the shortest wait. Ride quality is driving both variables at the same time.

The right way to test this is within each ride: when the same attraction has a longer wait on a given day, does fun actually go down?

In [ ]:
within_ride = (
    rides.groupby('ride_name')
         .apply(lambda g: pd.Series({
             'wait_fun_corr': g['actual_wait_min'].corr(g['fun_score']),
             'avg_wait': g['actual_wait_min'].mean(),
             'avg_fun': g['fun_score'].mean(),
             'n': len(g)
         }))
         .reset_index()
         .sort_values('avg_wait', ascending=False)
)

print('Within-ride correlation between wait time and fun score:')
print('(All should be negative — longer waits hurt fun even on great rides)')
print()
print(within_ride[['ride_name', 'wait_fun_corr', 'avg_wait', 'avg_fun', 'n']].round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: aggregate (misleading)
ride_avgs = rides.groupby('ride_name')[['actual_wait_min', 'fun_score']].mean().reset_index()
sns.scatterplot(data=ride_avgs, x='actual_wait_min', y='fun_score',
                s=100, ax=axes[0])
for _, row in ride_avgs.iterrows():
    axes[0].text(row['actual_wait_min'] + 0.5, row['fun_score'],
                 row['ride_name'].split(':')[0].split('the')[0][:20],
                 fontsize=7, alpha=0.8)
sns.regplot(data=ride_avgs, x='actual_wait_min', y='fun_score',
            scatter=False, color='red', ax=axes[0])
axes[0].set_title('Aggregate: Wait vs. Fun by Ride\n(Positive slope — misleading)', fontweight='bold')
axes[0].set_xlabel('Avg Actual Wait (min)')
axes[0].set_ylabel('Avg Fun Score')

# Right: within-ride correlations
within_ride_sorted = within_ride.sort_values('wait_fun_corr')
colors = ['#d73027' if x >= 0 else '#4575b4' for x in within_ride_sorted['wait_fun_corr']]
axes[1].barh(within_ride_sorted['ride_name'], within_ride_sorted['wait_fun_corr'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.9, linestyle='--')
axes[1].set_title('Within-Ride: Wait vs. Fun Correlation\n(All negative — longer waits reduce fun)', fontweight='bold')
axes[1].set_xlabel('Pearson r')
axes[1].set_ylabel('')
axes[1].tick_params(axis='y', labelsize=8)

plt.suptitle('Ride Quality Confounding — Aggregate vs. Within-Ride Analysis',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** The aggregate positive slope is entirely a confounding artifact. Better rides draw more guests, which drives up both wait times and fun scores together. When you look within a single ride, longer waits consistently reduce fun across the board. Every within-ride correlation is negative. This is a version of Simpson's Paradox that shows up regularly in theme park data, and catching it is what separates a surface-level analysis from one that is actually useful.

---
## 10. Ticket Type Segmentation

**Question:** Do annual passholders, standard guests, and park hoppers actually behave differently?

In [ ]:
ticket_profile = (
    df.groupby('ticket_type')
      .agg(
          n_events=('guest_id', 'count'),
          avg_party_size=('party_size', 'mean'),
          avg_fun=('fun_score', 'mean'),
          avg_frustration=('frustration_score', 'mean'),
          avg_arrival_hour=('hour', 'mean'),
      )
      .round(2)
)
print(ticket_profile)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ticket_order = ['standard', 'park_hopper', 'annual_pass']
palette = sns.color_palette('muted', 3)

# Arrival hour distribution
for i, (ticket, color) in enumerate(zip(ticket_order, palette)):
    subset = df[df['ticket_type'] == ticket]['hour']
    axes[0].hist(subset, bins=range(9, 22), alpha=0.65, label=ticket,
                 color=color, density=True, edgecolor='white')
axes[0].set_title('Arrival Hour by Ticket Type', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Fun score boxplot
sns.boxplot(data=df, x='ticket_type', y='fun_score', order=ticket_order,
            palette='muted', ax=axes[1])
axes[1].set_title('Fun Score by Ticket Type', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Fun Score')

# Frustration score boxplot
sns.boxplot(data=df, x='ticket_type', y='frustration_score', order=ticket_order,
            palette='muted', ax=axes[2])
axes[2].set_title('Frustration Score by Ticket Type', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('Frustration Score')

plt.suptitle('Guest Behavior by Ticket Type', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** Annual passholders arrive later and travel in smaller groups. They know the park well enough to skip the morning rush. Park hoppers show up latest, usually after spending the morning somewhere else. Standard guests trend toward early arrival and pack in more events, which makes sense given they are making the most of a single-day ticket. Treating all three groups the same in a crowd model would miss real behavioral differences that affect how density builds throughout the day.

---
## 11. Festival Comparison

**Question:** Does which festival is running change guest experience outcomes in a measurable way?

In [ ]:
festival_summary = (
    df.groupby('festival')[['fun_score', 'frustration_score', 'crowd_level']]
      .agg(['mean', 'median', 'std'])
      .round(2)
)
print(festival_summary)

In [ ]:
festival_order = (
    df.groupby('festival')['fun_score'].median()
      .sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=df, x='festival', y='fun_score', order=festival_order,
            palette='muted', ax=axes[0])
axes[0].set_title('Fun Score by Festival', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25)

sns.boxplot(data=df, x='festival', y='frustration_score', order=festival_order,
            palette='muted', ax=axes[1])
axes[1].set_title('Frustration Score by Festival', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=25)

sns.boxplot(data=df, x='festival', y='crowd_level', order=festival_order,
            palette='muted', ax=axes[2])
axes[2].set_title('Crowd Level by Festival', fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=25)

plt.suptitle('Guest Experience by Festival Context', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** Festival context shifts all three metrics. Food and Wine draws the heaviest crowds, and that pressure shows up in higher frustration medians. Flower and Garden runs lighter and produces tighter, more consistent fun scores. Festival of the Arts sits in the middle on both crowd density and score variability. Some of this reflects the events themselves, and some of it reflects who shows up, since audience composition varies meaningfully across festivals.

---
## 12. Correlation Summary

A quick look at how all numeric variables relate to each other.

In [ ]:
numeric_cols = [
    'posted_wait_min', 'actual_wait_min', 'wait_delta',
    'ride_duration_min', 'crowd_level', 'temperature_f',
    'party_size', 'frustration_score', 'fun_score'
]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## 13. Conclusion

This notebook looked at roughly 80,000 simulated EPCOT guest events across zones, rides, festivals, and ticket types. Here is what stood out:

**Congestion and flow.** Guest density follows a predictable arc peaking around midday. World Discovery and World Nature sustain the widest inflow windows. World Showcase traffic builds through the afternoon as guests move through pavilions in sequence.

**Wait times.** A small number of rides drive most of the wait burden. High-variability rides like Guardians and Frozen are the hardest to plan around. Posted times are slightly overstated on average, but the guests most at risk are the ones who hit right-tail waits that exceed what the sign said.

**Experience metrics.** Wait time is the strongest driver of frustration, and crowd level adds an independent layer on top of that. The aggregate positive correlation between waits and fun is a confounding artifact, not a real effect. Within any individual ride, longer waits reduce fun consistently.

**Segmentation.** Annual passholders, park hoppers, and standard guests show meaningfully different arrival patterns and group sizes. Aggregating across ticket types hides differences that matter for operational planning.

**Festivals.** Food and Wine drives the highest crowds and the highest frustration. Flower and Garden is the most consistent.

---

### Potential Next Steps
- Build a regression or gradient boosting model to predict frustration score from wait delta, crowd level, festival, and ticket type
- Cluster guests by movement pattern and experience profile to identify behavioral archetypes
- Forecast zone-level congestion by hour to support staffing and routing decisions
- Add ticket type and festival filters to the Power BI dashboard for interactive exploration